# E2 — Horizontal circle (anisotropy)

`e2_circle.py`. e_F is split into its **radial** part (along the centripetal acceleration: the inertia
error) and its **tangential** part (along the motion: friction, which no model has). Expected: M4's radial
e_F oscillates at twice the circle frequency, M5's stays flat.

In [ ]:
# Runs to analyse: run directories, session files (exp/<exp>/sessions/*.txt, written by the script) or a
# folder of runs. Default: every e2 run on the robot.
RUNS = ["~/data/franka/rim-delays/robot/e2"]
RECOMPUTE = False  # True: recompute the metrics from the data instead of reading analysis/metrics.json

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()  # experiments/rim-delays/exp
sys.path[:0] = [str(HERE), str(HERE.parent)]
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

import exp_metrics
import plots as P
from common import collect, load_table

%matplotlib inline
pd.set_option("display.precision", 3)

runs = collect(RUNS)
df = load_table(runs, exp_metrics.e2 if RECOMPUTE else None)
print(f"{len(df)} runs")

In [ ]:
df[["run", "method", "rtt", "eF_rms", "eF_radial_mean", "eF_radial_rms", "eF_tangential_mean", "eF_tangential_rms",
    "eF_z_rms", "angle_f_deg", "angle_Fosc_deg", "E_end_J"]].sort_values(["rtt", "method"])

## e_F against the circle phase
Radial (left) and tangential (right); one row per delay.

In [ ]:
delays = [d for d in P.DELAYS if d in set(df.rtt.map(P.delay_key))]
fig, axes = plt.subplots(len(delays), 2, figsize=(11, 3.2 * len(delays)), squeeze=False, sharex=True)
for i, T in enumerate(delays):
    sub = df[df.rtt.map(P.delay_key) == T]
    for _, row in sub.iterrows():
        m = P.metrics_of(row.dir)
        for j, key in enumerate(("eF_radial_by_phase", "eF_tangential_by_phase")):
            y = np.array([np.nan if v is None else v for v in m[key]])
            axes[i, j].plot(m["phase_deg"], y, color=P.color(row.method), label=P.LABELS.get(row.method, row.method) if (i, j) == (0, 0) else None)
    axes[i, 0].set_ylabel(f"T = {T}\nradial e_F [N]")
    axes[i, 1].set_ylabel("tangential e_F [N]")
for ax in axes[-1]:
    ax.set_xlabel("circle phase [deg]")
fig.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.04))
fig.tight_layout()

## Mean radial and tangential e_F, and the force direction
Angles: from the centripetal direction to the felt force and to the robot's force (0° = parallel).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.4))
for ax, y, title in ((axes[0], "eF_radial_rms", "RMS radial e_F [N]"), (axes[1], "eF_tangential_mean", "mean tangential e_F [N]")):
    P.by_delay(ax, df, y, label_methods=(ax is axes[0]))
    ax.set_title(title, loc="left", fontsize=10)
P.by_delay(axes[2], df, "angle_Fosc_deg", label_methods=False)
axes[2].set_title("angle: centripetal → robot force [deg]", loc="left", fontsize=10)
fig.legend(loc="upper center", ncol=3, bbox_to_anchor=(0.5, 1.08))
fig.tight_layout()